## 2.3 构建三层 ConvLSTM 网络

把 2.2 单层的 `self.convLSTM_cell` 改成 `cell1 / cell2 / cell3`。
当前层的 $h_t$ 当作后一层的 $x_t$。构造函数仍是 `ConvLSTM(input_dim, hidden_dim, kernel_size)`。

In [ ]:
import torch
import torch.nn as nn


class CLSTM_cell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size):
        super(CLSTM_cell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        self.kernel_size = kernel_size
        self.padding = (kernel_size[0] // 2, kernel_size[1] // 2)
        self.conv = nn.Conv2d(
            in_channels=input_dim + hidden_dim,
            out_channels=4 * hidden_dim,
            kernel_size=kernel_size,
            padding=self.padding,
        )

    def forward(self, input_tensor, hidden_state):
        h_cur, c_cur = hidden_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, (h_next, c_next)

In [ ]:
class ConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size):
        super(ConvLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        # 单层：self.convLSTM_cell = CLSTM_cell(input_dim, hidden_dim, kernel_size)
        # 三层：第 1 层读 x_t，第 2、3 层读上一层的 h_t
        self.cell1 = CLSTM_cell(input_dim, hidden_dim, kernel_size)
        self.cell2 = CLSTM_cell(hidden_dim, hidden_dim, kernel_size)
        self.cell3 = CLSTM_cell(hidden_dim, hidden_dim, kernel_size)

    def _init_hidden(self, batch_size, image_size):
        height, width = image_size

        def zeros(cell):
            # device=self.conv.weight.device: 选择设备
            device = cell.conv.weight.device
            return (
                torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
            )

        # 初始化返回的隐状态与后面使用的隐状态组成成分一样；
        # 当前层的 h_t 会当作后一层的 x_t，所以三层各自一份 (h, c)
        return [zeros(self.cell1), zeros(self.cell2), zeros(self.cell3)]

    def forward(self, inputs=None, hidden_state=None):
        # 取出图片的数据，供下面初始化使用
        s, b, _, h, w = inputs.size()
        if hidden_state is None:
            # 初始化 hidden_state，利用后面和 lstm 单元中的初始化函数
            hidden_state = self._init_hidden(batch_size=b, image_size=(h, w))

        def run_layer(cell, seq, state):
            output_inner = []
            for t in range(s):
                # 每一个时间步都更新 hidden_state 中的 h, c
                # 当前时间步的输出 hidden_state，又作为下一时间步的输入 hidden_state
                hy, state = cell(seq[t, ...], state)
                output_inner.append(hy)
            return torch.stack(output_inner), state

        # 当前层的 h_t 会当作后一层的 x_t 作为输入使用
        y1, s1 = run_layer(self.cell1, inputs, hidden_state[0])
        y2, s2 = run_layer(self.cell2, y1, hidden_state[1])
        y3, s3 = run_layer(self.cell3, y2, hidden_state[2])
        return y3, [s1, s2, s3]

In [ ]:
# 用法与单层完全相同
seq_len, batch, channels, height, width = 10, 2, 1, 32, 32
model = ConvLSTM(input_dim=channels, hidden_dim=16, kernel_size=3)
x = torch.randn(seq_len, batch, channels, height, width)
y, states = model(x)
print(y.shape)              # (10, 2, 16, 32, 32)
print(len(states))          # 3
print(states[-1][0].shape)  # (2, 16, 32, 32)